In [ ]:

import sys
sys.path.insert(0, '/home/amtramos/Dropbox/pessoal/learn_python/Data/')
from yahoofinance import YF 
import pandas as pd
y = YF()
y.init_param()
y.chg_interval('2000-01-01', pd.Timestamp.today().strftime('%Y-%m-%d'))    


listTicker = ['AMZN', 'MSFT', 'AAPL', 'NVDA', 'GOOGL', 'SPY']

df = pd.DataFrame()
for ticker in listTicker:
    dftemp = y.get_data(ticker)
    dftemp = dftemp[['close']]
    dftemp.columns = [ticker]
    dftemp.index = [pd.to_datetime(x.strftime('%Y-%m-%d')) for x in dftemp.index]
    df = pd.concat([df,dftemp],axis=1)
    


In [ ]:
# df = pdr.get_data_yahoo("GOVT SPY BTC-USD BRL=X AAPL GOLD", start="2017-01-01", end="2020-01-01")
dfRetX = df.copy()
dfRetX = dfRetX.fillna(method='ffill')

dfRetX = np.log(dfRetX) - np.log(dfRetX.shift(1))
# dfRetX = dfRetX.pct_change(1)
dfRetX = dfRetX.dropna(how='any',axis=0)
dfRetX

In [ ]:
dfRetX10 = dfRetX.rolling(21).sum()
# dfRetX10 = dfRetX10.ewm(com=0.4).mean()
dfRetX10 = dfRetX10.rolling(14).mean()
dfRetX10.loc['2024':,:].plot()


In [ ]:

dfSignal = dfRetX10.copy()
for col in dfRetX10.columns:
    dftemp = dfRetX10.loc[:,col] - dfRetX10.loc[:,col].shift(14)
    mask = dftemp > 0
    dfSignal.loc[mask,col] = 1
    dfSignal.loc[~mask,col] = 0
    
pd.concat([dfSignal.loc['2024':,'NVDA'],dfRetX10.loc['2024':,'NVDA']],axis=1).plot()


In [ ]:
dfSignal

In [ ]:
srW = dfSignal.apply(lambda x : abs(x)).sum(axis=1).apply(lambda x : 1 if x == 0 else x)
for col in  dfSignal.columns:
    dfSignal.loc[:,col] = dfSignal.loc[:,col]/srW

In [ ]:
dfSignal.loc['2024':,:].plot()


In [ ]:
dfSignal.corr()

In [ ]:


dfRetY0 = (dfSignal*dfRetX).sum(axis=1)
dfRetY0 = pd.DataFrame(dfRetY0,columns = ['PORTFOLIO'])
dfRetY0

In [ ]:
(dfRetX + 1).cumprod()

In [ ]:
listStocks = ['AAPL','AMZN','GOOGL','MSFT']
dfRetXY = dfRetX.loc[:,listStocks].copy()

In [ ]:
np.pi

In [ ]:


x = np.linspace(0,10*np.pi,dfRetXY.shape[0])
ysin = (0.4+0.3*np.sin(x))/1.6
ysin90 = (0.4-0.3*np.sin(x))/1.6
ysinF = (0.4+0.3*np.sin(x/2))/1.6
ysin90F = (0.4-0.3*np.sin(x/2))/1.6

pd.DataFrame([ysin + ysin90 + ysinF + ysin90F]).T.plot()


In [ ]:
dfWtg = pd.DataFrame([ysin,ysin90,ysinF,ysin90F],index = dfRetXY.columns, columns = dfRetXY.index).T
dfWtg.plot()

In [ ]:
dfWtg.corr()


In [ ]:

dfRetY = (dfRetXY*dfWtg).sum(axis=1)
dfRetY = pd.DataFrame(dfRetY,columns = ['PORTFOLIO'])
dfRetY

In [ ]:
dftemp = pd.concat([(dfRetY0+1).cumprod(), (dfRetY+1).cumprod()],axis=1)
dftemp.columns = columns=['mom','sin']
dftemp.plot()

In [ ]:

Y = dfRetY.loc[:,['PORTFOLIO']]
X = dfRetX.copy()
# X = sm.add_constant(X) # manually add a constant here
# X = sm.add_constant(X)
model = sm.OLS(Y,X)
results = model.fit()
results.summary()

In [ ]:
results.params.values.sum()

In [ ]:
from sklearn.linear_model import LassoLarsIC
from sklearn.linear_model import Lasso

modelLassoLarsBIC = LassoLarsIC(criterion='bic',fit_intercept=False).fit(X.iloc[:,1:],Y.values.ravel())
cteAlphBIC = modelLassoLarsBIC.alpha_
cteAlphBIC 

In [ ]:
modelLasso = Lasso(alpha=cteAlphBIC*2,fit_intercept=True,normalize=False,max_iter=1000).fit(X.iloc[:,1:],Y.values.ravel())
modelLasso.coef_